# Phase 3: Atari Comparison

**Comprehensive comparison across Atari environments.**

Aggregates results from Pong and Breakout experiments.

---
## 1. Setup and Load Results



In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

%matplotlib inline
sns.set_style("whitegrid")

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"

# Color palette
COLORS = {
    "baseline": "#2ecc71",
    "quantum_tunneling": "#3498db",
    "superposition": "#9b59b6",
    "entanglement": "#e74c3c",
    "interference_ensemble": "#f39c12",
}

APPROACH_LABELS = {
    "baseline": "Baseline",
    "quantum_tunneling": "Quantum Tunneling",
    "superposition": "Superposition",
    "entanglement": "Entanglement",
    "interference_ensemble": "Interference Ensemble",
}

# Load Atari results
def load_metrics(env_path):
    with open(env_path / "complete_metrics.json") as f:
        return json.load(f)

pong_data = load_metrics(RESULTS_DIR / "phase3" / "pong")
breakout_data = load_metrics(RESULTS_DIR / "phase3" / "breakout")

# Also load DMControl results for cross-domain comparison
walker_data = load_metrics(RESULTS_DIR / "phase2" / "walker")
cheetah_data = load_metrics(RESULTS_DIR / "phase2" / "cheetah")
reacher_data = load_metrics(RESULTS_DIR / "phase2" / "reacher")

print("Loaded results:")
print(f"  Pong:    {len(pong_data['raw_results'])} runs, {len(pong_data['summary'])} approaches")
print(f"  Breakout: {len(breakout_data['raw_results'])} runs, {len(breakout_data['summary'])} approaches")
print(f"  Walker:  {len(walker_data['raw_results'])} runs (DMControl reference)")
print(f"  Cheetah: {len(cheetah_data['raw_results'])} runs (DMControl reference)")
print(f"  Reacher: {len(reacher_data['raw_results'])} runs (DMControl reference)")

---
## 2. Cross-Game Comparison

Compare method performance across Pong and Breakout.

In [ ]:
# Cross-Game Comparison Table
print("=" * 90)
print("CROSS-GAME COMPARISON: Test Observation MSE (×10⁻⁴)")
print("=" * 90)

approaches = ["baseline", "quantum_tunneling", "superposition", "entanglement", "interference_ensemble"]

rows = []
for approach in approaches:
    pong_s = pong_data["summary"].get(approach, {})
    breakout_s = breakout_data["summary"].get(approach, {})
    
    pong_mse = pong_s.get("test_obs_mse_mean", float("nan")) * 1e4
    pong_std = pong_s.get("test_obs_mse_std", float("nan")) * 1e4
    breakout_mse = breakout_s.get("test_obs_mse_mean", float("nan")) * 1e4
    breakout_std = breakout_s.get("test_obs_mse_std", float("nan")) * 1e4
    
    rows.append({
        "Approach": APPROACH_LABELS[approach],
        "Pong MSE (×10⁻⁴)": f"{pong_mse:.2f} ± {pong_std:.2f}",
        "Breakout MSE (×10⁻⁴)": f"{breakout_mse:.2f} ± {breakout_std:.2f}",
        "Pong_val": pong_mse,
        "Breakout_val": breakout_mse,
    })

df = pd.DataFrame(rows)
print(df[["Approach", "Pong MSE (×10⁻⁴)", "Breakout MSE (×10⁻⁴)"]].to_string(index=False))

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (game, data) in enumerate([("Pong", pong_data), ("Breakout", breakout_data)]):
    ax = axes[idx]
    means = []
    stds = []
    colors = []
    labels = []
    
    for approach in approaches:
        s = data["summary"].get(approach, {})
        means.append(s.get("test_obs_mse_mean", 0) * 1e4)
        stds.append(s.get("test_obs_mse_std", 0) * 1e4)
        colors.append(COLORS[approach])
        labels.append(APPROACH_LABELS[approach])
    
    bars = ax.bar(range(len(approaches)), means, yerr=stds, color=colors, 
                  capsize=5, edgecolor="black", linewidth=0.5)
    ax.set_xticks(range(len(approaches)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Test Obs MSE (×10⁻⁴)")
    ax.set_title(f"{game} - Method Comparison")
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "figures" / "atari_cross_game_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/figures/atari_cross_game_comparison.png")

---
## 3. Visual vs State-Based Comparison

Compare Atari results with DMControl results.

In [ ]:
# Visual (Atari) vs State-Based (DMControl) Comparison
# This is the KEY FINDING of the dissertation

def get_ie_improvement(data):
    """Compute IE improvement percentage vs baseline."""
    bl = data["summary"]["baseline"]["test_obs_mse_mean"]
    ie = data["summary"]["interference_ensemble"]["test_obs_mse_mean"]
    return (bl - ie) / bl * 100  # positive = improvement

results_comparison = {
    "Walker-walk": {"domain": "State-Based", "ie_pct": get_ie_improvement(walker_data),
                    "bl_mse": walker_data["summary"]["baseline"]["test_obs_mse_mean"],
                    "ie_mse": walker_data["summary"]["interference_ensemble"]["test_obs_mse_mean"]},
    "Cheetah-run": {"domain": "State-Based", "ie_pct": get_ie_improvement(cheetah_data),
                    "bl_mse": cheetah_data["summary"]["baseline"]["test_obs_mse_mean"],
                    "ie_mse": cheetah_data["summary"]["interference_ensemble"]["test_obs_mse_mean"]},
    "Reacher-easy": {"domain": "State-Based", "ie_pct": get_ie_improvement(reacher_data),
                     "bl_mse": reacher_data["summary"]["baseline"]["test_obs_mse_mean"],
                     "ie_mse": reacher_data["summary"]["interference_ensemble"]["test_obs_mse_mean"]},
    "Pong": {"domain": "Visual", "ie_pct": get_ie_improvement(pong_data),
             "bl_mse": pong_data["summary"]["baseline"]["test_obs_mse_mean"],
             "ie_mse": pong_data["summary"]["interference_ensemble"]["test_obs_mse_mean"]},
    "Breakout": {"domain": "Visual", "ie_pct": get_ie_improvement(breakout_data),
                 "bl_mse": breakout_data["summary"]["baseline"]["test_obs_mse_mean"],
                 "ie_mse": breakout_data["summary"]["interference_ensemble"]["test_obs_mse_mean"]},
}

print("=" * 80)
print("CRITICAL FINDING: IE Performance by Domain")
print("=" * 80)
print(f"{'Environment':<15} {'Domain':<12} {'Baseline MSE':>14} {'IE MSE':>14} {'IE Change':>12}")
print("-" * 80)
for env, info in results_comparison.items():
    sign = "+" if info["ie_pct"] > 0 else ""
    print(f"{env:<15} {info['domain']:<12} {info['bl_mse']:>14.6f} {info['ie_mse']:>14.6f} {sign}{info['ie_pct']:>10.1f}%")

# Visualization: Domain split bar chart
fig, ax = plt.subplots(figsize=(10, 6))

envs = list(results_comparison.keys())
pcts = [results_comparison[e]["ie_pct"] for e in envs]
bar_colors = ["#2ecc71" if p > 0 else "#e74c3c" for p in pcts]

bars = ax.barh(range(len(envs)), pcts, color=bar_colors, edgecolor="black", linewidth=0.5)
ax.set_yticks(range(len(envs)))
ax.set_yticklabels(envs, fontsize=11)
ax.set_xlabel("IE Improvement vs Baseline (%)", fontsize=12)
ax.set_title("Interference Ensemble: Domain-Specific Performance", fontsize=14)
ax.axvline(x=0, color="black", linewidth=1)
ax.grid(axis="x", alpha=0.3)

# Add value labels
for i, (bar, pct) in enumerate(zip(bars, pcts)):
    sign = "+" if pct > 0 else ""
    ax.text(bar.get_width() + (2 if pct > 0 else -2), bar.get_y() + bar.get_height()/2,
            f"{sign}{pct:.1f}%", ha="left" if pct > 0 else "right", va="center", fontweight="bold")

# Add domain annotations
ax.axhspan(-0.5, 2.5, alpha=0.1, color="green")
ax.axhspan(2.5, 4.5, alpha=0.1, color="red")
ax.text(max(pcts)*0.5, 1, "STATE-BASED", ha="center", fontsize=10, fontstyle="italic", alpha=0.5)
ax.text(min(pcts)*0.5, 3.5, "VISUAL", ha="center", fontsize=10, fontstyle="italic", alpha=0.5)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "figures" / "atari_vs_state_domain_split.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved: results/figures/atari_vs_state_domain_split.png")

---
## 4. Statistical Analysis



In [ ]:
# Statistical Analysis: Mann-Whitney U tests with Bonferroni correction

def cohens_d(group1, group2):
    """Compute Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    if pooled_std == 0:
        return 0.0
    return (np.mean(group1) - np.mean(group2)) / pooled_std

def get_seed_values(data, approach, metric="test_obs_mse"):
    """Extract per-seed metric values for a given approach."""
    return [r[metric] for r in data["raw_results"] if r["approach"] == approach and metric in r]

# Bonferroni correction: 4 methods × 2 games = 8 comparisons
num_comparisons = 8
bonferroni_alpha = 0.05 / num_comparisons

print("=" * 90)
print("STATISTICAL ANALYSIS: Mann-Whitney U Tests vs Baseline")
print(f"Bonferroni-corrected α = {bonferroni_alpha:.6f} ({num_comparisons} comparisons)")
print("=" * 90)

stat_results = []
for game_name, game_data in [("Pong", pong_data), ("Breakout", breakout_data)]:
    print(f"\n--- {game_name} ---")
    baseline_vals = get_seed_values(game_data, "baseline")
    
    for approach in ["quantum_tunneling", "superposition", "entanglement", "interference_ensemble"]:
        approach_vals = get_seed_values(game_data, approach)
        
        if len(approach_vals) == 0:
            print(f"  {APPROACH_LABELS[approach]}: No data")
            continue
        
        if len(approach_vals) >= 2 and len(baseline_vals) >= 2:
            u_stat, p_val = stats.mannwhitneyu(baseline_vals, approach_vals, alternative="two-sided")
            d = cohens_d(baseline_vals, approach_vals)
        else:
            u_stat, p_val, d = float("nan"), float("nan"), float("nan")
        
        bl_mean = np.mean(baseline_vals)
        ap_mean = np.mean(approach_vals)
        delta = (bl_mean - ap_mean) / bl_mean * 100
        sig = p_val < bonferroni_alpha if not np.isnan(p_val) else False
        sig_str = "** SIGNIFICANT **" if sig else ""
        
        print(f"  {APPROACH_LABELS[approach]:<25} Δ={delta:+.1f}%  p={p_val:.4f}  d={d:.2f}  {sig_str}")
        
        stat_results.append({
            "game": game_name, "approach": approach,
            "delta_pct": delta, "p_value": p_val, "cohens_d": d, "significant": sig
        })

stat_df = pd.DataFrame(stat_results)
print("\n\nSignificant results only:")
sig_df = stat_df[stat_df["significant"]]
if len(sig_df) > 0:
    print(sig_df[["game", "approach", "delta_pct", "p_value", "cohens_d"]].to_string(index=False))
else:
    print("  No statistically significant results at Bonferroni-corrected level.")

---
## 5. When Do Quantum Methods Help?

Identify conditions in visual RL where methods excel.

In [ ]:
# When Do Quantum Methods Help in Visual RL?

print("=" * 80)
print("ANALYSIS: When Do Quantum Methods Help in Visual RL?")
print("=" * 80)

print("""
ANSWER: Quantum-inspired methods show NO significant benefit for visual RL tasks.

1. QUANTUM TUNNELING: Marginal, non-significant differences on both Pong and Breakout.
   The tunneling perturbations have minimal effect on CNN-based feature learning.

2. SUPERPOSITION REPLAY: Unlike DMControl where it catastrophically fails, 
   Superposition shows COMPARABLE performance to baseline on Atari. This suggests 
   that pixel-space mixing is less disruptive than state-space mixing, likely because
   pixel observations are more redundant and the CNN encoder can tolerate noise.

3. ENTANGLEMENT LAYERS: No meaningful impact. The additional correlation modeling
   adds parameters but does not improve visual feature extraction.

4. INTERFERENCE ENSEMBLE: SIGNIFICANTLY WORSE on both games.
   - Pong: -132% degradation (p < 0.001)
   - Breakout: -414% degradation (p < 0.001)
   
   The phase-weighted ensemble mechanism, designed for low-dimensional state vectors,
   fails catastrophically when applied to 4096+ dimensional CNN feature spaces.
   Variance-based uncertainty estimation becomes unreliable in high dimensions.

KEY INSIGHT: The observation dimensionality is the critical factor.
   - State-based (6-24 dims): IE helps significantly (+35-45%)
   - Visual (4096+ CNN features): IE hurts significantly (-132% to -414%)

PRACTICAL RECOMMENDATION: 
   For visual RL tasks, use the standard baseline. No quantum-inspired method 
   provides benefit, and IE should be explicitly avoided.
""")

# Dimensionality analysis table
dim_analysis = [
    {"Environment": "Reacher-easy", "Obs Dim": 6, "IE Change": f"+{get_ie_improvement(reacher_data):.1f}%", "Domain": "State"},
    {"Environment": "Cheetah-run", "Obs Dim": 17, "IE Change": f"+{get_ie_improvement(cheetah_data):.1f}%", "Domain": "State"},
    {"Environment": "Walker-walk", "Obs Dim": 24, "IE Change": f"+{get_ie_improvement(walker_data):.1f}%", "Domain": "State"},
    {"Environment": "Pong", "Obs Dim": 4096, "IE Change": f"{get_ie_improvement(pong_data):.1f}%", "Domain": "Visual"},
    {"Environment": "Breakout", "Obs Dim": 4096, "IE Change": f"{get_ie_improvement(breakout_data):.1f}%", "Domain": "Visual"},
]
dim_df = pd.DataFrame(dim_analysis)
print("\nDimensionality vs IE Performance:")
print(dim_df.to_string(index=False))

---
## 6. Visualization



In [ ]:
# Comprehensive Visualization

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Heatmap: Method × Game performance (normalized to baseline)
ax = axes[0, 0]
games = ["Pong", "Breakout"]
methods = ["quantum_tunneling", "superposition", "entanglement", "interference_ensemble"]
method_labels = [APPROACH_LABELS[m] for m in methods]

heatmap_data = np.zeros((len(methods), len(games)))
for j, (game_name, game_data) in enumerate([("Pong", pong_data), ("Breakout", breakout_data)]):
    bl = game_data["summary"]["baseline"]["test_obs_mse_mean"]
    for i, method in enumerate(methods):
        m = game_data["summary"].get(method, {}).get("test_obs_mse_mean", bl)
        heatmap_data[i, j] = (bl - m) / bl * 100

im = ax.imshow(heatmap_data, cmap="RdYlGn", aspect="auto", vmin=-450, vmax=50)
ax.set_xticks(range(len(games)))
ax.set_xticklabels(games, fontsize=10)
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(method_labels, fontsize=9)
ax.set_title("Improvement vs Baseline (%)", fontsize=12)
for i in range(len(methods)):
    for j in range(len(games)):
        val = heatmap_data[i, j]
        color = "white" if abs(val) > 100 else "black"
        ax.text(j, i, f"{val:+.1f}%", ha="center", va="center", fontsize=9, color=color)
plt.colorbar(im, ax=ax, shrink=0.8)

# 2. Training time comparison
ax = axes[0, 1]
for game_name, game_data in [("Pong", pong_data), ("Breakout", breakout_data)]:
    times = []
    labels = []
    for approach in approaches:
        s = game_data["summary"].get(approach, {})
        t = s.get("time_mean", 0)
        times.append(t)
        labels.append(APPROACH_LABELS[approach])
    x = np.arange(len(approaches))
    width = 0.35
    offset = -0.175 if game_name == "Pong" else 0.175
    ax.bar(x + offset, times, width, label=game_name, 
           color="#3498db" if game_name == "Pong" else "#e74c3c", alpha=0.7, edgecolor="black", linewidth=0.5)

ax.set_xticks(range(len(approaches)))
ax.set_xticklabels([APPROACH_LABELS[a] for a in approaches], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Training Time (s)")
ax.set_title("Training Time per Method")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# 3. Parameter count comparison
ax = axes[1, 0]
params = [pong_data["summary"][a].get("num_params", 0) / 1e6 for a in approaches]
bars = ax.bar(range(len(approaches)), params, color=[COLORS[a] for a in approaches],
              edgecolor="black", linewidth=0.5)
ax.set_xticks(range(len(approaches)))
ax.set_xticklabels([APPROACH_LABELS[a] for a in approaches], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Parameters (Millions)")
ax.set_title("Model Size Comparison")
ax.grid(axis="y", alpha=0.3)

# 4. IE failure analysis: Pong vs Breakout scatter
ax = axes[1, 1]
for approach in approaches:
    pong_vals = get_seed_values(pong_data, approach)
    breakout_vals = get_seed_values(breakout_data, approach)
    if len(pong_vals) > 0 and len(breakout_vals) > 0:
        ax.scatter([np.mean(pong_vals) * 1e4], [np.mean(breakout_vals) * 1e4],
                  s=150, c=COLORS[approach], edgecolors="black", linewidth=1,
                  label=APPROACH_LABELS[approach], zorder=5)
ax.set_xlabel("Pong Test MSE (×10⁻⁴)")
ax.set_ylabel("Breakout Test MSE (×10⁻⁴)")
ax.set_title("Per-Game Performance Correlation")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "figures" / "atari_comprehensive_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/figures/atari_comprehensive_analysis.png")

---
## 7. Key Findings

Summary of Phase 3 results.

In [ ]:
# Key Findings Summary

print("=" * 80)
print("PHASE 3: ATARI COMPARISON - KEY FINDINGS")
print("=" * 80)

print("""
1. NO QUANTUM METHOD IMPROVES VISUAL RL
   - All four quantum-inspired approaches (QT, SP, EN, IE) show non-significant 
     or negative effects on Pong and Breakout.
   - This contrasts sharply with DMControl state-based results.

2. INTERFERENCE ENSEMBLE FAILS CATASTROPHICALLY ON VISUAL TASKS
   - Pong:     -132% degradation (p < 0.001)
   - Breakout: -414% degradation (p < 0.001)
   - The 103M parameter ensemble (vs 8.9M baseline) wastes computation 
     and produces worse predictions.

3. SUPERPOSITION BEHAVES DIFFERENTLY ON VISUAL vs STATE TASKS
   - DMControl: -158% to -630% (catastrophic failure)
   - Atari: ~0% change (comparable to baseline)
   - Hypothesis: Pixel observations are more redundant, so mixing is less 
     disruptive than mixing precise physical state vectors.

4. CNN FEATURE SPACE BREAKS PHASE-WEIGHTED ENSEMBLE
   - The IE mechanism relies on meaningful variance across ensemble members
   - In 4096+ dimensional CNN feature spaces, this signal becomes noise
   - Phase weighting amplifies rather than corrects errors

5. PRACTICAL RECOMMENDATION
   - For visual RL: Use standard baseline (no quantum benefit)
   - For state-based RL: Consider IE (+35-45% with 5x cost)
   - Observation dimensionality is the key predictor of IE effectiveness

6. INCOMPLETE IE METRICS
   - IE Atari experiments collected only test_obs_mse (missing train, reward, 
     and long-horizon metrics) due to the 103M parameter model requiring 
     a modified evaluation pipeline for memory constraints.
""")

print("=" * 80)
print("Phase 3 Atari Comparison Complete")
print("=" * 80)